# Tool-use eval — `sft/eval_tool_use.py`

Runs the **same tool loop the chatbot serves** over three legs of never-trained
(val-side) questions, greedy decoding:

| leg | prompt | input | expectation |
|---|---|---|---|
| A | canonical tool prompt | hard math (tool_use val) | call rate high, pass@1 strictly between 0 and 100% |
| B | canonical tool prompt | trivial math / prose (tool_negative val) | **false-call rate low** |
| C | *no* prompt | the same hard math | call rate **0%** |

Model-written python runs in the sandbox on this VM (network blocked). Needs a GPU runtime.

In [ ]:
# Mount Drive + paths
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
CKPT    = f'{SYNAPSE_DIR}/sft_checkpoints/v3_15source/sft_best.pth'
RESULTS = f'{SYNAPSE_DIR}/tool_eval_results'
N       = 100          # records per leg (3 legs); ~5-10 min on an A100/T4
for p in (CKPT, f'{SYNAPSE_DIR}/datasets_sft/tool_use/tool_use_raw.jsonl',
          f'{SYNAPSE_DIR}/datasets_sft/tool_negative/tool_negative_raw.jsonl',
          f'{SYNAPSE_DIR}/tokenizer_out/tokenizer.json'):
    print(('  OK ' if os.path.exists(p) else 'MISS ') + p)

In [ ]:
# Clone/pull repo
import subprocess
REPO_DIR = '/content/synapse_repo'
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/ajencinas/synapse.git', REPO_DIR], check=True)
assert os.path.isfile(os.path.join(REPO_DIR, 'sft', 'eval_tool_use.py'))

In [ ]:
# Deps + GPU check + BRAVE key (leg D does LIVE searches — train == inference)
!pip -q install tokenizers sympy
import torch
assert torch.cuda.is_available(), 'no GPU — Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0))

# Brave key from Colab Secrets (🔑 sidebar → add BRAVE_API_KEY, enable notebook access).
BRAVE = ''
try:
    from google.colab import userdata
    BRAVE = userdata.get('BRAVE_API_KEY') or ''
except Exception:
    pass
LEGS = 'A,B,C,D' if BRAVE else 'A,B,C'
print('leg D (search):', 'ENABLED' if BRAVE else 'SKIPPED — no BRAVE_API_KEY secret')

In [ ]:
# Run the eval (a results JSON with every trace lands on Drive)
import time
out = f'{RESULTS}/tool_eval_{int(time.time())}.json'
env = f'SYNAPSE_DIR={SYNAPSE_DIR}' + (f' BRAVE_API_KEY={BRAVE}' if BRAVE else '')
cmd = (f'cd {REPO_DIR} && {env} python sft/eval_tool_use.py '
       f'--ckpt "{CKPT}" --n {N} --legs {LEGS} --output "{out}" --quiet')
print(cmd.replace(BRAVE, '****') if BRAVE else cmd)
!{cmd}
import os
assert os.path.exists(out), 'eval did not write results — read the error above (nothing stale is shown)'

In [ ]:
# Inspect THIS run's traces (cell 4's exact output file — never a stale glob)
import json
r = json.load(open(out))
for L, leg in r['legs'].items():
    print('\n#####', L, leg['title'], json.dumps(leg['metrics']))
    for it in leg['items'][:3]:
        print('  Q:', it['question'][:120])
        for c in it['calls']: print('  CALL:', json.dumps(c)[:160])
        for t in it['results']: print('  RESULT:', t[:120])
        print('  FINAL:', it['final'][-160:], '| gold', it['gold'], '| pass', it['pass'], '| status', it['status'])

## Reading the numbers
- **A pass@1** is the headline: can it solve held-out math *using* the tool.
  0% means the loop is broken; 100% means the questions leaked.
- **B call rate** is the false-call rate. v1 had no negatives at all (it would
  have been ~100%); the whole point of `tool_negative` is to bring this down.
- **C call rate must be 0**: without the tool prompt the model has never seen
  `<|tool_call|>` in context. Anything above 0 means prompt→behavior is leaking.
- `malformed_rate` > a few % means the JSON format didn't fully take.